## Q1 — Which donors are likely to donate again?

Shared data preparation appears below. Then: time-aware CV and logistic regression on `repeat_within_h`.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Data: folder `lighthouse_csv_files` next to this notebook (under ML pipelines/)
_data_dir = Path("lighthouse_csv_files")
_donations_csv = _data_dir / "donations.csv"
_donation_allocations_csv = _data_dir / "donation_allocations.csv"
_supporters_csv = _data_dir / "supporters.csv"
donations = pd.read_csv(_donations_csv.resolve())
allocations = pd.read_csv(_donation_allocations_csv.resolve())
supporters = pd.read_csv(_supporters_csv.resolve())

supporters["first_donation_date"] = pd.to_datetime(
    supporters["first_donation_date"], errors="coerce"
)
supporters["created_at"] = pd.to_datetime(supporters["created_at"], errors="coerce")

donations["donation_date"] = pd.to_datetime(donations["donation_date"], errors="coerce")
allocations["allocation_date"] = pd.to_datetime(allocations["allocation_date"], errors="coerce")

donations["donation_year"] = donations["donation_date"].dt.year
donations["donation_month"] = donations["donation_date"].dt.month

# Future: days_since_last_donation — group by supporter_id, sort by donation_date, diff in days vs prior row.

for col in ("amount", "estimated_value"):
    donations[col] = pd.to_numeric(donations[col], errors="coerce")
allocations["amount_allocated"] = pd.to_numeric(allocations["amount_allocated"], errors="coerce")

_donation_value = donations["amount"].where(donations["amount"].notna(), donations["estimated_value"])
donations["donation_value"] = _donation_value
donations["is_zero_value_donation"] = _donation_value.eq(0)
donations["is_negative_value_donation"] = _donation_value.lt(0)

donations.head()

In [ ]:
donations = donations[
    donations["donation_value"].notna()
    & (donations["donation_value"] > 0)
] # remove zero and negative value donations
donations = donations.dropna(subset=["supporter_id", "donation_date"])
# drop rows with missing supporter_id or donation_date
donations = donations.sort_values(
    ["supporter_id", "donation_date"]
)
# sort by supporter_id and donation_date
donations["next_donation_date"] = donations.groupby("supporter_id")[
    "donation_date"
].shift(-1)
donations["days_to_next_donation"] = (
    donations["next_donation_date"] - donations["donation_date"]
).dt.days
donations["days_since_last_donation"] = (
    donations["donation_date"] - donations.groupby("supporter_id")["donation_date"].shift(1)
).dt.days


In [ ]:
# Allocation-level features (one row per donation_id)
_by_area = (
    allocations.groupby(["donation_id", "program_area"], as_index=False)["amount_allocated"]
    .sum()
)
_total_allocated = allocations.groupby("donation_id")["amount_allocated"].sum().rename("total_amount_allocated")
_num_alloc = allocations.groupby("donation_id").size().rename("num_allocations")

_area_with_total = _by_area.merge(_total_allocated.reset_index(), on="donation_id", validate="many_to_one")
_area_with_total["share_of_allocated"] = _area_with_total["amount_allocated"] / _area_with_total[
    "total_amount_allocated"
].replace(0, np.nan)
_concentration = _area_with_total.groupby("donation_id")["share_of_allocated"].max().rename(
    "allocation_concentration"
)

alloc_features = (
    pd.concat([_num_alloc, _total_allocated, _concentration], axis=1)
    .reset_index()
    .merge(donations[["donation_id", "donation_value"]], on="donation_id", how="left", validate="one_to_one")
)

# Share of donation_value covered by allocations (1.0 == 100%); NaN if donation_value is 0 or missing
alloc_features["pct_donation_allocated"] = alloc_features["total_amount_allocated"] / alloc_features[
    "donation_value"
].replace(0, np.nan)

# True when summed allocations match donation_value within tolerance (fully allocated / "sums to 100%")
_alloc_rtol = 1e-5
_alloc_atol = 0.01
_alloc_tot = alloc_features["total_amount_allocated"]
_dv = alloc_features["donation_value"]
alloc_features["is_allocation_fully_specified"] = (
    _dv.notna()
    & _dv.gt(0)
    & np.isclose(_alloc_tot, _dv, rtol=_alloc_rtol, atol=_alloc_atol)
)

alloc_features.head()

### Model base (`model_df`)

# One row per `donation_id`: donations + allocation features + supporter attributes (no email/phone/name PII for modeling).

In [ ]:
_alloc_merge = alloc_features.drop(columns=["donation_value"], errors="ignore")

model_df = donations.merge(
    _alloc_merge,
    on="donation_id",
    how="left",
    validate="one_to_one",
)

_supporter_for_model = supporters.drop(
    columns=[
        "display_name",
        "organization_name",
        "first_name",
        "last_name",
        "email",
        "phone",
    ],
    errors="ignore",
)

model_df = model_df.merge(
    _supporter_for_model,
    on="supporter_id",
    how="left",
    validate="many_to_one",
)

model_df = model_df.sort_values(["supporter_id", "donation_date"]).reset_index(drop=True)

print(
    "Row counts — donations (filtered):",
    len(donations),
    "| alloc_features:",
    len(alloc_features),
    "| supporters:",
    len(supporters),
    "| model_df:",
    len(model_df),
)

model_df["is_non_monetary"] = model_df["donation_type"].ne("Monetary")
_nm_cum = model_df.groupby("supporter_id")["is_non_monetary"].cumsum()
model_df["prior_non_monetary_count"] = (
    _nm_cum - model_df["is_non_monetary"].astype(int)
).clip(lower=0)
model_df["prior_donation_index"] = model_df.groupby("supporter_id").cumcount()
model_df["next_donation_type"] = model_df.groupby("supporter_id")["donation_type"].shift(-1)

model_df.head(3)

### Outcomes (prediction horizon)

Use **H = 180** days. **`repeat_within_h`**: 1 if the supporter’s **next** donation is within H days; 0 if we observe H days from `donation_date` to `dataset_end` with **no** further donation; **NaN** if censored (no next row and follow-up &lt; H). Models use only **eligible** rows (not censored).


In [ ]:
H = 180
_dataset_end = model_df["donation_date"].max()

_has_next = model_df["next_donation_date"].notna()
_window_days = (_dataset_end - model_df["donation_date"]).dt.days

model_df["repeat_within_h_eligible"] = _has_next | (_window_days >= H)

model_df["repeat_within_h"] = np.nan
model_df.loc[_has_next, "repeat_within_h"] = (
    model_df.loc[_has_next, "days_to_next_donation"] <= H
).astype(float)
model_df.loc[~_has_next & (_window_days >= H), "repeat_within_h"] = 0.0

# Q4-related: next gift within H is monetary (NaN if censored / no next in window)
model_df["next_monetary_within_h"] = np.nan
_m_next = _has_next & (model_df["days_to_next_donation"] <= H)
model_df.loc[_m_next, "next_monetary_within_h"] = model_df.loc[
    _m_next, "next_donation_type"
].eq("Monetary").astype(float)
model_df.loc[~_has_next & (_window_days >= H), "next_monetary_within_h"] = 0.0
model_df["next_monetary_eligible"] = model_df["repeat_within_h_eligible"]

print("dataset_end", _dataset_end.date())
print(
    "repeat_within_h eligible:",
    int(model_df["repeat_within_h_eligible"].sum()),
    "| labeled:",
    int(model_df["repeat_within_h"].notna().sum()),
    "| positive rate (labeled):",
    f"{model_df.loc[model_df['repeat_within_h'].notna(), 'repeat_within_h'].mean():.3f}",
)

### Q1 — Which donors are likely to donate again?

**Target:** `repeat_within_h` (binary). **Validation:** time-aware **expanding-window** CV (`TimeSeriesSplit`): within each fold, no training row is dated **after** any test row—aligned with training on history and evaluating on a future slice. Same supporter may still appear in both sides; interpret as associational, not causal. **Model:** balanced logistic regression on donation- and supporter-level features. After CV, the pipeline is **refit** on the full time-ordered labeled set for inspection or reuse.

In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import average_precision_score, classification_report, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

_numeric_for_q1 = [
    "donation_value",
    "days_since_last_donation",
    "prior_non_monetary_count",
    "prior_donation_index",
    "num_allocations",
    "pct_donation_allocated",
    "allocation_concentration",
    "is_recurring",
]
_categorical_for_q1 = [
    "donation_type",
    "channel_source",
    "campaign_name",
    "supporter_type",
    "region",
    "country",
    "relationship_type",
    "status",
    "acquisition_channel",
]

labeled_q1 = model_df[
    model_df["repeat_within_h_eligible"] & model_df["repeat_within_h"].notna()
].copy()
labeled_q1_sorted = labeled_q1.sort_values(
    ["donation_date", "donation_id"], kind="mergesort"
).reset_index(drop=True)

X_q1 = labeled_q1_sorted[_numeric_for_q1 + _categorical_for_q1].copy()
for _col in _numeric_for_q1:
    X_q1[_col] = pd.to_numeric(X_q1[_col], errors="coerce")
X_q1[_categorical_for_q1] = X_q1[_categorical_for_q1].fillna("Unknown")
y_q1 = labeled_q1_sorted["repeat_within_h"].astype(int)

_prep_q1 = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            _numeric_for_q1,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    (
                        "oh",
                        OneHotEncoder(handle_unknown="ignore", max_categories=30),
                    ),
                ]
            ),
            _categorical_for_q1,
        ),
    ]
)
C_grid = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]
for C in C_grid:
    print(f"\n=== Testing C = {C} ===")

    _q1_base = Pipeline(
        steps=[
            ("prep", _prep_q1),
            (
                "lr",
                LogisticRegression(
                    C=C,
                    max_iter=5000,
                    class_weight="balanced",
                    solver="saga",
                    random_state=42,
                ),
            ),
        ]
    )

    _fold_aucs = []
    _fold_pr_aucs = []

    for _fold_i, (_train_idx, _test_idx) in enumerate(_tscv_q1.split(X_q1), start=1):
        _est = clone(_q1_base)
        _X_tr, _X_te = X_q1.iloc[_train_idx], X_q1.iloc[_test_idx]
        _y_tr, _y_te = y_q1.iloc[_train_idx], y_q1.iloc[_test_idx]

        if _y_te.nunique() < 2:
            continue

        _est.fit(_X_tr, _y_tr)
        _proba_te = _est.predict_proba(_X_te)[:, 1]

        _fold_pr_aucs.append(average_precision_score(_y_te, _proba_te))
        _fold_aucs.append(roc_auc_score(_y_te, _proba_te))

    print(
        "Mean PR-AUC:",
        f"{np.mean(_fold_pr_aucs):.3f}",
        "| Mean ROC-AUC:",
        f"{np.mean(_fold_aucs):.3f}",
    )




_q1_clf = Pipeline(
    steps=[
        ("prep", _prep_q1),
        (
            "lr",
            LogisticRegression(
                C=0.01,
                max_iter=5000,
                class_weight="balanced",
                solver="saga",
                random_state=42,
            ),
        ),
    ]
)

_q1_clf.fit(X_q1, y_q1)
print("Final Q1 model (C=0.01) refit on full labeled set, n =", len(X_q1))

In [ ]:
feature_names = (
    _q1_clf.named_steps["prep"]
    .get_feature_names_out()
)

coefs = _q1_clf.named_steps["lr"].coef_[0]

coef_df = (
    pd.DataFrame({"feature": feature_names, "coef": coefs})
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values("abs_coef", ascending=False)
)

coef_df.head(15)

Conclusion:
We tested this model as if we had deployed it at multiple points in time. 
It consistently identified likely repeat donors, and it improved with time